# Fashion Image Retrieval — Colab 실행 노트북

이미지 임베딩으로 **같은 상품 찾기** (DeepFashion In-Shop).
Baseline(zero-shot) → 실패 분석 → 개선 실험 순서로 위에서부터 실행하세요.

> ⚠️ **먼저 런타임 → 런타임 유형 변경 → 하드웨어 가속기 = T4 GPU** 로 설정!

## 0. GPU 확인 + 패키지 설치

In [ ]:
!nvidia-smi -L
!pip -q install faiss-cpu transformers datasets kagglehub
import torch; print('CUDA available:', torch.cuda.is_available())

## 1. 코드(`src/`) 가져오기
본인 GitHub 레포 URL로 바꾸세요. (아직 레포가 없으면, 왼쪽 파일창에 `src/` 폴더를 업로드해도 됩니다.)

In [ ]:
import os, sys
REPO_URL = "https://github.com/YOUR_ID/fashion-image-retrieval.git"   # ← YOUR_ID만 본인 계정으로 교체
if not os.path.exists('fashion-image-retrieval'):
    !git clone $REPO_URL
%cd /content/fashion-image-retrieval
sys.path.append('/content/fashion-image-retrieval')

## 2. Google Drive 마운트 (임베딩 캐시 저장)
세션이 끊겨도 임베딩(.npy)은 Drive에 남아 재평가가 순식간입니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CACHE = '/content/drive/MyDrive/retrieval_cache'
os.makedirs(CACHE, exist_ok=True)
print('cache:', CACHE)

## 3. In-Shop 데이터 다운로드 (Kaggle, 자동)
- 최초 1회 Kaggle 토큰 필요: kaggle.com → 우측상단 프로필 → **Settings → API → Create New API Token** → 받은 `kaggle.json`을 Colab 파일창에 업로드하거나 아래가 뜨면 붙여넣기.
- 손으로 파일 옮길 필요 없이, 이 셀이 받아서 압축까지 풉니다 (~7GB, 몇 분).

In [ ]:
import kagglehub
DATA_ROOT = kagglehub.dataset_download("hserdaraltan/deepfashion-inshop-clothes-retrieval")
print('DATA_ROOT =', DATA_ROOT)

## 4. 데이터 로드 + 서브샘플
무료 Colab에 맞춰 상품 수를 줄입니다 (숫자는 자유롭게 조정).
- `query`: 500개 상품
- `gallery`: 최대 8,000장 (정답은 유지, distractor만 무작위)

In [ ]:
from src import data
splits = data.load_inshop(DATA_ROOT)
query, gallery, train = splits['query'], splits['gallery'], splits['train']
print('raw  ->  query:%d  gallery:%d  train:%d' % (len(query), len(gallery), len(train)))

query   = data.subsample_items(query, n_items=500, seed=0)
gallery = data.cap_gallery(query, gallery, max_gallery=8000, seed=0)
print('use  ->  query:%d  gallery:%d' % (len(query), len(gallery)))

## 5. Baseline 임베딩 — DINOv2 (zero-shot)
검색용 self-supervised 백본. CLS 토큰을 L2 정규화해서 사용.

In [ ]:
import numpy as np
from src import embed
E = embed.Embedder('dinov2')
g_vecs = E.encode(gallery.paths, batch_size=64, desc='gallery')
q_vecs = E.encode(query.paths,   batch_size=64, desc='query')
np.save(f'{CACHE}/g_dinov2.npy', g_vecs)
np.save(f'{CACHE}/q_dinov2.npy', q_vecs)
print('embeddings:', g_vecs.shape, q_vecs.shape)

## 6. FAISS 검색 + Recall@k (baseline 숫자)

In [ ]:
from src import metrics
nrel = metrics.n_relevant_per_query(query.item_ids, gallery.item_ids)
sims, idx = metrics.search(g_vecs, q_vecs, topk=50)
base = metrics.evaluate(idx, query.item_ids, gallery.item_ids, ks=(1,5,10), n_relevant=nrel)
results = {'dinov2_zeroshot': base}
print('DINOv2 zero-shot:', {k: round(v,3) for k,v in base.items()})

## 7. ★ 실패 갤러리 — 왜 틀렸나 (이 프로젝트의 심장)
top-5에 정답이 없는 query를 그리드로 저장하고 자동 태깅.
- `same_category_confusion`: 같은 카테고리인데 다른 상품 (색/형태 과의존 의심)
- `cross_category`: 아예 다른 카테고리 (배경/전역특징에 끌림 의심)

**저장된 이미지를 직접 눈으로 보고** 유형을 확정하세요. 그게 다음 개선의 근거가 됩니다.

In [ ]:
from src import failures
tags, n_fail = failures.build_gallery(idx, sims, query, gallery,
                                      out_dir='results/failures', k=5, n=60)
print(f'실패 query {n_fail}개 / 자동 태그 분포:', tags)

from IPython.display import Image as IPImage, display
import glob
for p in sorted(glob.glob('results/failures/*.png'))[:6]:
    display(IPImage(p))

---
# Day 2 — 개선 실험
각 실험을 baseline과 같은 프로토콜로 재평가하고 채택/기각을 수치로 남깁니다.

## 8. 백본 비교 — CLIP / ResNet50 (가설: self-sup 백본이 검색에 유리)

In [ ]:
for bk in ['clip', 'resnet50']:
    Eb = embed.Embedder(bk)
    gv = Eb.encode(gallery.paths, desc=f'{bk} gallery')
    qv = Eb.encode(query.paths,   desc=f'{bk} query')
    _, ix = metrics.search(gv, qv, topk=50)
    results[f'{bk}_zeroshot'] = metrics.evaluate(ix, query.item_ids, gallery.item_ids,
                                                 ks=(1,5,10), n_relevant=nrel)
    print(bk, {k: round(v,3) for k,v in results[f'{bk}_zeroshot'].items()})
    del Eb

## 9. 전처리 실험 — center-crop (가설: 배경 제거로 상품에 집중)

In [ ]:
gc = E.encode(gallery.paths, center_crop=True, desc='gallery crop')
qc = E.encode(query.paths,   center_crop=True, desc='query crop')
_, ixc = metrics.search(gc, qc, topk=50)
results['dinov2_centercrop'] = metrics.evaluate(ixc, query.item_ids, gallery.item_ids,
                                                ks=(1,5,10), n_relevant=nrel)
print('center-crop:', {k: round(v,3) for k,v in results['dinov2_centercrop'].items()})

## 10. Fine-tune — projection head (가설: 도메인 적응으로 검색 향상)
얼린 DINOv2 임베딩 위에 작은 head를 supervised-contrastive로 학습.
train 상품 1,000개만 사용 → 몇 분이면 수렴.

In [ ]:
from src import finetune
train_s = data.subsample_items(train, n_items=1000, seed=0)
tr_vecs = E.encode(train_s.paths, batch_size=64, desc='train')
np.save(f'{CACHE}/train_dinov2.npy', tr_vecs)

head = finetune.train_head(tr_vecs, train_s.item_ids, dim=tr_vecs.shape[1],
                           P=16, K=4, steps=600, lr=1e-3, temp=0.1)
g_ft = finetune.apply_head(head, g_vecs)
q_ft = finetune.apply_head(head, q_vecs)
_, ixf = metrics.search(g_ft, q_ft, topk=50)
results['dinov2_finetune'] = metrics.evaluate(ixf, query.item_ids, gallery.item_ids,
                                              ks=(1,5,10), n_relevant=nrel)
print('fine-tune:', {k: round(v,3) for k,v in results['dinov2_finetune'].items()})

## 11. 결과 표 저장 (→ README/포폴에 붙이기)

In [ ]:
import pandas as pd
df = pd.DataFrame(results).T[['recall@1','recall@5','recall@10','mAP@10']].round(4)
os.makedirs('results', exist_ok=True)
df.to_markdown('results/metrics.md')
df.to_csv('results/metrics.csv')
print(df)
print('\n저장: results/metrics.md, results/failures/*.png')
print('이제 로컬에서 git add results/ 후 커밋/푸시하세요 (커밋은 본인 계정으로!).')

In [22]:
import os, sys, subprocess
os.chdir('/content')
REPO = '/content/fashion-image-retrieval'
subprocess.run(['rm', '-rf', REPO])
r = subprocess.run(['git', 'clone',
                    'https://github.com/UPTOUOY/fashion-image-retrieval.git', REPO],
                   capture_output=True, text=True)
print(r.stderr.strip() or 'clone OK')
print('src:', os.listdir(REPO + '/src') if os.path.isdir(REPO + '/src') else '❌ 없음')
sys.path.insert(0, REPO)
os.chdir(REPO)
print('cwd =', os.getcwd())

Cloning into '/content/fashion-image-retrieval'...
src: ['data.py', 'failures.py', 'finetune_backbone.py', 'embed.py', '__init__.py', 'finetune.py', 'metrics.py']
cwd = /content/fashion-image-retrieval


In [23]:
import sys
for m in list(sys.modules):
    if m == 'src' or m.startswith('src.'):
        del sys.modules[m]
from src import finetune_backbone as fb, data, metrics

train_s = data.subsample_items(train, n_items=1500, seed=0)

# ★ dinov2-base(768d)로 백본 fine-tune — T4에서 ~12~18분
model_b = fb.finetune_backbone(train_s.paths, train_s.item_ids,
                               epochs=3, P=8, K=4, steps_per_epoch=200,
                               lr=2e-5, unfreeze_blocks=4,
                               model_name='facebook/dinov2-base')

gbb = fb.encode_backbone(model_b, gallery.paths, batch_size=48, desc='gallery(base)')
qbb = fb.encode_backbone(model_b, query.paths,   batch_size=48, desc='query(base)')
_, ixbb = metrics.search(gbb, qbb, topk=50)
results['dinov2base_backbone_ft'] = metrics.evaluate(ixbb, query.item_ids, gallery.item_ids,
                                                     ks=(1,5,10), n_relevant=nrel)

print('base ft  :', {k: round(v,3) for k,v in results['dinov2base_backbone_ft'].items()})
print('small ft :', {k: round(v,3) for k,v in results['dinov2_backbone_ft'].items()})
print('baseline :', {k: round(v,3) for k,v in results['dinov2_zeroshot'].items()})

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

학습 파라미터: 28.36M (unfreeze_blocks=4)
  epoch 0 | step    0 | loss 3.6455
  epoch 0 | step   50 | loss 1.6355
  epoch 0 | step  100 | loss 1.7043
  epoch 0 | step  150 | loss 1.5138
  epoch 1 | step  200 | loss 1.3187
  epoch 1 | step  250 | loss 1.6680
  epoch 1 | step  300 | loss 1.5389
  epoch 1 | step  350 | loss 1.3771
  epoch 2 | step  400 | loss 1.5634
  epoch 2 | step  450 | loss 1.4296
  epoch 2 | step  500 | loss 2.1297
  epoch 2 | step  550 | loss 1.3880


query(base): 100%|██████████| 11/11 [00:14<00:00,  1.32s/it]

base ft  : {'recall@1': 0.874, 'recall@5': 0.954, 'recall@10': 0.968, 'mAP@10': 0.6}
small ft : {'recall@1': 0.852, 'recall@5': 0.946, 'recall@10': 0.962, 'mAP@10': 0.553}
baseline : {'recall@1': 0.492, 'recall@5': 0.664, 'recall@10': 0.726, 'mAP@10': 0.2}


In [24]:
import sys, numpy as np, torch
for m in list(sys.modules):
    if m=='src' or m.startswith('src.'): del sys.modules[m]
from src import finetune_backbone as fb, data, metrics

# 더 많은 데이터 (1500 → 3000 상품)
train_s = data.subsample_items(train, n_items=3000, seed=0)
print('train imgs:', len(train_s))

# dinov2-base · 6에폭 · 뒤 6블록 학습 (강화)
model_big = fb.finetune_backbone(train_s.paths, train_s.item_ids,
                                 model_name='facebook/dinov2-base',
                                 epochs=6, P=8, K=4, steps_per_epoch=250,
                                 lr=2e-5, unfreeze_blocks=6, size=224)

gbig = fb.encode_backbone(model_big, gallery.paths, batch_size=48, size=224, desc='gallery(big)')
qbig = fb.encode_backbone(model_big, query.paths,   batch_size=48, size=224, desc='query(big)')

# 임베딩 Drive에 저장 (세션 끊겨도 결과 보존)
np.save(f'{CACHE}/g_strong.npy', gbig); np.save(f'{CACHE}/q_strong.npy', qbig)

_, ixbig = metrics.search(gbig, qbig, topk=50)
results['dinov2base_strong'] = metrics.evaluate(ixbig, query.item_ids, gallery.item_ids,
                                                ks=(1,5,10), n_relevant=nrel)
print('strong   :', {k: round(v,3) for k,v in results['dinov2base_strong'].items()})
print('base ft  :', {k: round(v,3) for k,v in results['dinov2base_backbone_ft'].items()})
print('baseline :', {k: round(v,3) for k,v in results['dinov2_zeroshot'].items()})

train imgs: 24097


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

학습 파라미터: 42.54M (unfreeze_blocks=6)
  epoch 0 | step    0 | loss 3.9515
  epoch 0 | step   50 | loss 1.7545
  epoch 0 | step  100 | loss 1.3905
  epoch 0 | step  150 | loss 1.3601
  epoch 0 | step  200 | loss 1.4323
  epoch 1 | step  250 | loss 1.3802
  epoch 1 | step  300 | loss 1.3677
  epoch 1 | step  350 | loss 1.7097
  epoch 1 | step  400 | loss 1.6694
  epoch 1 | step  450 | loss 1.3223
  epoch 2 | step  500 | loss 1.3858
  epoch 2 | step  550 | loss 1.5691
  epoch 2 | step  600 | loss 1.7661
  epoch 2 | step  650 | loss 1.4081
  epoch 2 | step  700 | loss 1.2719
  epoch 3 | step  750 | loss 1.4417
  epoch 3 | step  800 | loss 1.2564
  epoch 3 | step  850 | loss 1.4954
  epoch 3 | step  900 | loss 1.3448
  epoch 3 | step  950 | loss 1.5118
  epoch 4 | step 1000 | loss 1.4171
  epoch 4 | step 1050 | loss 1.6258
  epoch 4 | step 1100 | loss 1.2459
  epoch 4 | step 1150 | loss 1.5782
  epoch 4 | step 1200 | loss 1.6517
  epoch 5 | step 1250 | loss 1.4921
  epoch 5 | step 1300 | loss

query(big): 100%|██████████| 11/11 [00:14<00:00,  1.35s/it]

strong   : {'recall@1': 0.884, 'recall@5': 0.962, 'recall@10': 0.98, 'mAP@10': 0.621}
base ft  : {'recall@1': 0.874, 'recall@5': 0.954, 'recall@10': 0.968, 'mAP@10': 0.6}
baseline : {'recall@1': 0.492, 'recall@5': 0.664, 'recall@10': 0.726, 'mAP@10': 0.2}


In [19]:
# src 캐시 비우고 최신 코드 로드
import sys
for m in list(sys.modules):
    if m == 'src' or m.startswith('src.'):
        del sys.modules[m]
from src import finetune_backbone as fb, data, metrics

# train 상품 1500개 (약 8천장)
train_s = data.subsample_items(train, n_items=1500, seed=0)
print('train imgs:', len(train_s))

# ★ 백본 직접 학습 (뒤 4블록, 3에폭) — 5~7분
model = fb.finetune_backbone(train_s.paths, train_s.item_ids,
                             epochs=3, P=8, K=4, steps_per_epoch=200,
                             lr=2e-5, unfreeze_blocks=4)

# 학습된 백본으로 재임베딩 → 평가 (2~3분)
gb = fb.encode_backbone(model, gallery.paths, batch_size=64, desc='gallery(ft)')
qb = fb.encode_backbone(model, query.paths,   batch_size=64, desc='query(ft)')
_, ixb = metrics.search(gb, qb, topk=50)
results['dinov2_backbone_ft'] = metrics.evaluate(ixb, query.item_ids, gallery.item_ids,
                                                 ks=(1,5,10), n_relevant=nrel)

print('backbone ft:', {k: round(v,3) for k,v in results['dinov2_backbone_ft'].items()})
print('head-only  :', {k: round(v,3) for k,v in results['dinov2_finetune'].items()})
print('baseline   :', {k: round(v,3) for k,v in results['dinov2_zeroshot'].items()})

train imgs: 11797


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

학습 파라미터: 7.10M (unfreeze_blocks=4)
  epoch 0 | step    0 | loss 3.1276
  epoch 0 | step   50 | loss 1.8679
  epoch 0 | step  100 | loss 1.8023
  epoch 0 | step  150 | loss 1.5556
  epoch 1 | step  200 | loss 1.4144
  epoch 1 | step  250 | loss 1.6462
  epoch 1 | step  300 | loss 1.5584
  epoch 1 | step  350 | loss 1.5256
  epoch 2 | step  400 | loss 1.6152
  epoch 2 | step  450 | loss 1.5471
  epoch 2 | step  500 | loss 2.2212
  epoch 2 | step  550 | loss 1.5801


query(ft): 100%|██████████| 8/8 [00:09<00:00,  1.18s/it]

backbone ft: {'recall@1': 0.852, 'recall@5': 0.946, 'recall@10': 0.962, 'mAP@10': 0.553}
head-only  : {'recall@1': 0.696, 'recall@5': 0.886, 'recall@10': 0.93, 'mAP@10': 0.408}
baseline   : {'recall@1': 0.492, 'recall@5': 0.664, 'recall@10': 0.726, 'mAP@10': 0.2}


In [20]:
from src import failures
fa = failures.find_failures(ixb, query.item_ids, gallery.item_ids, k=5)
print(f'top-5 실패: baseline 168개 → backbone_ft {len(fa)}개')

top-5 실패: baseline 168개 → backbone_ft 27개


In [21]:
import pandas as pd, os
order = ['dinov2_zeroshot','dinov2_centercrop','dinov2_finetune','dinov2_backbone_ft']
df = pd.DataFrame({k: results[k] for k in order if k in results}).T[['recall@1','recall@5','recall@10','mAP@10']].round(4)
os.makedirs('results', exist_ok=True)
df.to_markdown('results/metrics.md'); df.to_csv('results/metrics.csv')
print(df)

                    recall@1  recall@5  recall@10  mAP@10
dinov2_zeroshot        0.492     0.664      0.726  0.1998
dinov2_centercrop      0.494     0.662      0.726  0.1997
dinov2_finetune        0.696     0.886      0.930  0.4082
dinov2_backbone_ft     0.852     0.946      0.962  0.5532
